Install Libraries for Semantic Search

In [ ]:
%%capture
!pip install -q sentence-transformers rank-bm25

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

# Sample corporate/tech dataset
corpus = [
    "How to hard-reset your iPhone 13 if the touch screen is completely frozen or unresponsive.",  # Doc 0
    "Troubleshooting guide for iOS updates failing on newer Apple mobile devices.",               # Doc 1
    "The new Samsung Galaxy S26 Ultra features an advanced generative AI camera system.",         # Doc 2
    "Steps to recover a lost Google Pixel account recovery phrase or authentication token.",      # Doc 3
    "Fixing Wi-Fi connectivity drops and network configuration errors on Apple MacBook laptops.", # Doc 4
    "Why is my smartphone battery draining so quickly? Top power optimization tips."              # Doc 5
]

print(f"Loaded database with {len(corpus)} technical documents.")

Loaded database with 6 technical documents.


In [ ]:
# Tokenize corpus for BM25 processing
tokenized_corpus = [doc.lower().split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_search(query, top_n=3):
    """Executes keyword lookup and returns top document indices and raw scores."""

    tokenized_query = query.lower().split(" ")
    scores = bm25.get_scores(tokenized_query)

    # Sort indices by score descending
    top_indices = np.argsort(scores)[::-1][:top_n]

    # Return list of tuples: (doc_id, score)
    return [(idx, scores[idx]) for idx in top_indices if scores[idx] > 0]

# Quick verification test
print("Sparse test search for 'iPhone 13':", sparse_search("iPhone 13"))

Sparse test search for 'iPhone 13': [(np.int64(0), np.float64(2.3996477957205307))]


In [ ]:
# Initialize a lightweight, high-performance bi-encoder model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate dense vector embeddings for the entire corpus
corpus_embeddings = embedding_model.encode(
    corpus,
    convert_to_tensor=True
)

def dense_search(query, top_n=5):
    """
    Executes semantic search via cosine similarity
    and returns sorted indices and scores.
    """

    query_embedding = embedding_model.encode(
        query,
        convert_to_tensor=True
    )

    # Calculate cosine similarity matrix against all corpus documents
    cos_scores = util.cos_sim(
        query_embedding,
        corpus_embeddings
    )[0]

    # Sort top elements
    top_results = np.argsort(
        cos_scores.cpu().numpy()
    )[::-1][:top_n]

    return [
        (int(idx), float(cos_scores[idx]))
        for idx in top_results
    ]

# Quick verification test
print(
    "Dense test search for 'phone model':",
    dense_search("phone model")
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense test search for 'phone model': [(2, 0.3248240351676941), (0, 0.22676901519298553), (5, 0.19839924573898315), (1, 0.18481627106666565), (3, 0.15603062510490417)]


In [ ]:
def reciprocal_rank_fusion(sparse_results, dense_results, k=60, top_n=3):
    """
    Fuses rankings from separate systems using the Reciprocal Rank Fusion formula.
    Inputs are expected to be lists of tuples: (doc_id, score) sorted by relevance.
    """

    rrf_scores = {}

    # Process sparse rankings
    for rank, (doc_id, _) in enumerate(sparse_results):
        # rank starts at 0, so rank position is rank + 1
        rrf_scores[doc_id] = (
            rrf_scores.get(doc_id, 0.0)
            + 1.0 / (k + (rank + 1))
        )

    # Process dense rankings
    for rank, (doc_id, _) in enumerate(dense_results):
        rrf_scores[doc_id] = (
            rrf_scores.get(doc_id, 0.0)
            + 1.0 / (k + (rank + 1))
        )

    # Sort documents based on their combined RRF metrics
    fused_rankings = sorted(
        rrf_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return fused_rankings[:top_n]

In [ ]:
def hybrid_search_engine(query, top_n=3):
    """
    Orchestrates both retrieval systems and fuses the output.
    """

    # 1. Gather deep recall arrays from both engines
    sparse_res = sparse_search(query, top_n=10)
    dense_res = dense_search(query, top_n=10)

    # 2. Fuse the rankings using RRF
    hybrid_res = reciprocal_rank_fusion(
        sparse_res,
        dense_res,
        k=60,
        top_n=top_n
    )

    # 3. Print pretty outputs for comparison
    print(f"\n=== TARGET QUERY: {query} ===")

    for rank, (doc_id, rrf_score) in enumerate(hybrid_res):
        print(f"\n[Rank {rank + 1}] (RRF Score: {rrf_score:.4f})")
        print(f"Document #{doc_id}: {corpus[doc_id]}")

    return hybrid_res

In [ ]:
# ----------------------------------------------------
# Execution Test 1: The Keyword Trap
# Query uses exact keyword terms from Doc 0,
# but conceptual theme of Doc 1
# ----------------------------------------------------
hybrid_search_engine("Apple mobile device issues")


# ----------------------------------------------------
# Execution Test 2: Multi-Concept Search
# Query matches semantic context of Doc 5
# but specific tech hardware in Doc 4
# ----------------------------------------------------
hybrid_search_engine("Macbook laptop battery optimization")


=== TARGET QUERY: Apple mobile device issues ===

[Rank 1] (RRF Score: 0.0328)
Document #1: Troubleshooting guide for iOS updates failing on newer Apple mobile devices.

[Rank 2] (RRF Score: 0.0318)
Document #4: Fixing Wi-Fi connectivity drops and network configuration errors on Apple MacBook laptops.

[Rank 3] (RRF Score: 0.0161)
Document #5: Why is my smartphone battery draining so quickly? Top power optimization tips.

=== TARGET QUERY: Macbook laptop battery optimization ===

[Rank 1] (RRF Score: 0.0328)
Document #5: Why is my smartphone battery draining so quickly? Top power optimization tips.

[Rank 2] (RRF Score: 0.0323)
Document #4: Fixing Wi-Fi connectivity drops and network configuration errors on Apple MacBook laptops.

[Rank 3] (RRF Score: 0.0159)
Document #2: The new Samsung Galaxy S26 Ultra features an advanced generative AI camera system.


[(np.int64(5), 0.03278688524590164),
 (np.int64(4), 0.03225806451612903),
 (2, 0.015873015873015872)]

#Extension Task

Ground Truth Dataset

In [ ]:
evaluation_queries = [
    {
        "query": "iPhone frozen screen",
        "relevant_docs": [0]
    },
    {
        "query": "Apple mobile update problems",
        "relevant_docs": [1]
    },
    {
        "query": "Samsung AI camera",
        "relevant_docs": [2]
    },
    {
        "query": "recover Pixel authentication token",
        "relevant_docs": [3]
    },
    {
        "query": "MacBook WiFi issue",
        "relevant_docs": [4]
    },
    {
        "query": "phone battery optimization",
        "relevant_docs": [5]
    }
]

print(f"Loaded {len(evaluation_queries)} evaluation queries")

Loaded 6 evaluation queries


MRR(Mean Reciporcal Rank) Calculation

In [ ]:
def calculate_mrr(results, relevant_docs):
    """
    Returns reciprocal rank for a single query.
    """

    for rank, (doc_id, _) in enumerate(results, start=1):
        if doc_id in relevant_docs:
            return 1.0 / rank

    return 0.0

NDCG Calculation

In [ ]:
def dcg(relevance_scores):
    """
    Discounted Cumulative Gain
    """
    score = 0

    for i, rel in enumerate(relevance_scores):
        score += rel / np.log2(i + 2)

    return score


def calculate_ndcg(results, relevant_docs, k=5):
    """
    Computes NDCG@k
    """

    predicted_relevance = []

    for doc_id, _ in results[:k]:
        predicted_relevance.append(
            1 if doc_id in relevant_docs else 0
        )

    ideal_relevance = sorted(
        predicted_relevance,
        reverse=True
    )

    actual_dcg = dcg(predicted_relevance)
    ideal_dcg = dcg(ideal_relevance)

    if ideal_dcg == 0:
        return 0

    return actual_dcg / ideal_dcg

Hybrid Search Evaluator

In [ ]:
def evaluate_hybrid_search():

    mrr_scores = []
    ndcg_scores = []

    print("=" * 80)
    print("HYBRID SEARCH EVALUATION")
    print("=" * 80)

    for test_case in evaluation_queries:

        query = test_case["query"]
        relevant_docs = test_case["relevant_docs"]

        sparse_res = sparse_search(query, top_n=10)
        dense_res = dense_search(query, top_n=10)

        fused_results = reciprocal_rank_fusion(
            sparse_res,
            dense_res,
            k=60,
            top_n=10
        )

        mrr = calculate_mrr(
            fused_results,
            relevant_docs
        )

        ndcg = calculate_ndcg(
            fused_results,
            relevant_docs,
            k=5
        )

        mrr_scores.append(mrr)
        ndcg_scores.append(ndcg)

        print(f"\nQuery: {query}")
        print(f"Relevant Docs: {relevant_docs}")
        print(f"MRR: {mrr:.4f}")
        print(f"NDCG@5: {ndcg:.4f}")

    avg_mrr = np.mean(mrr_scores)
    avg_ndcg = np.mean(ndcg_scores)

    print("\n" + "=" * 80)
    print("FINAL RESULTS")
    print("=" * 80)

    print(f"Average MRR    : {avg_mrr:.4f}")
    print(f"Average NDCG@5 : {avg_ndcg:.4f}")

    return avg_mrr, avg_ndcg

Evaluation

In [ ]:
avg_mrr, avg_ndcg = evaluate_hybrid_search()

HYBRID SEARCH EVALUATION

Query: iPhone frozen screen
Relevant Docs: [0]
MRR: 1.0000
NDCG@5: 1.0000

Query: Apple mobile update problems
Relevant Docs: [1]
MRR: 1.0000
NDCG@5: 1.0000

Query: Samsung AI camera
Relevant Docs: [2]
MRR: 1.0000
NDCG@5: 1.0000

Query: recover Pixel authentication token
Relevant Docs: [3]
MRR: 1.0000
NDCG@5: 1.0000

Query: MacBook WiFi issue
Relevant Docs: [4]
MRR: 1.0000
NDCG@5: 1.0000

Query: phone battery optimization
Relevant Docs: [5]
MRR: 1.0000
NDCG@5: 1.0000

FINAL RESULTS
Average MRR    : 1.0000
Average NDCG@5 : 1.0000
